In [1]:
# ============================================================
# MEMORY-EFFICIENT GROUP ERPAC AGGREGATION + 4-ROI PLOTTING
# ============================================================
#
# This version NEVER constructs a master subject-level long dataframe.
#
# It reads one participant-condition .pkl at a time and updates
# online (Welford) group statistics directly on the ERPAC matrices.
#
# Memory use therefore depends mainly on the number of final
# group-level maps, not on the total number of subject rows.
#
# Outputs:
#   - aggregated CSV + Parquet per group × condition
#   - one 2×2 ROI figure per group × condition × coupling
#   - fixed colour scale per coupling across ALL groups/conditions/ROIs
#
# ============================================================

from pathlib import Path
import pickle
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

ERPAC_INPUT_DIR = Path(
    r"F:\# study 2\eeg_data\erpac\PCA\erpac_roi_tc"
)

OUTPUT_ROOT = (
    ERPAC_INPUT_DIR
    / "group_averages_streaming"
)

AGG_DATA_DIR = (
    OUTPUT_ROOT
    / "aggregated_data"
)

FIG_DIR = (
    OUTPUT_ROOT
    / "figures"
)

AGG_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# SETTINGS
# ============================================================

TASKS = [
    "FTT",
    "DeCRAT",
]

ROI_ORDER = [
    "M1",
    "PMC",
    "S1",
    "SMA",
]

DPI = 300

# Fixed scale per coupling across ALL groups/conditions/ROIs.
#
# Robust scale:
GLOBAL_SCALE_PERCENTILES = (
    2,
    98,
)

# Set to None if you want literal global min/max:
# GLOBAL_SCALE_PERCENTILES = None


# ============================================================
# HELPERS
# ============================================================

def clean_optional(value):
    if value is None:
        return "NA"

    value = str(value)

    if value.lower() in {
        "none",
        "nan",
        "na",
        "",
    }:
        return "NA"

    return value


def safe_filename(text):
    return (
        str(text)
        .replace(" ", "_")
        .replace("/", "-")
        .replace("\\", "-")
        .replace(":", "-")
    )


def parse_condition_from_filename(
    file_path,
    sub,
):
    """
    Parse filenames like:

    s1_pac_sub01_FTT_plan_pca_roi_tc_erpac_results.pkl

    s1_pac_sub01_DeCRAT_plan_baseline_pca_roi_tc_erpac_results.pkl

    s1_pac_sub01_DeCRAT_go_adaptation_early_pca_roi_tc_erpac_results.pkl
    """

    suffix = (
        "_pca_roi_tc_erpac_results.pkl"
    )

    name = file_path.name

    if not name.endswith(
        suffix
    ):
        raise ValueError(
            f"Unexpected filename:\n{name}"
        )

    core = name[
        :-len(suffix)
    ]

    prefix = (
        f"{sub}_"
    )

    if not core.startswith(
        prefix
    ):
        raise ValueError(
            f"Filename does not start with subject ID {sub}:\n{name}"
        )

    condition = core[
        len(prefix):
    ]

    parts = condition.split(
        "_"
    )

    task = parts[0]
    stage = parts[1]

    block = "NA"
    adaptation_split = "NA"

    if task == "DeCRAT":

        if len(parts) >= 3:
            block = parts[2]

        if (
            block == "adaptation"
            and len(parts) >= 4
        ):
            adaptation_split = (
                parts[3]
            )

    elif task == "FTT":
        pass

    else:
        raise ValueError(
            f"Unknown task in filename:\n{name}"
        )

    return {
        "task": task,
        "task_stage": stage,
        "task_block": block,
        "adaptation_split":
            adaptation_split,
        "condition": condition,
    }


def find_pkl_files():

    if not ERPAC_INPUT_DIR.exists():
        raise FileNotFoundError(
            f"Input directory does not exist:\n"
            f"{ERPAC_INPUT_DIR}"
        )

    files = sorted(
        ERPAC_INPUT_DIR.rglob(
            "*_pca_roi_tc_erpac_results.pkl"
        )
    )

    files = [
        path
        for path in files
        if OUTPUT_ROOT not in path.parents
    ]

    if not files:

        print(
            "No matching ERPAC files found."
        )

        print(
            "\nExample .pkl files present:"
        )

        for path in list(
            ERPAC_INPUT_DIR.rglob(
                "*.pkl"
            )
        )[:20]:
            print(
                path
            )

        raise FileNotFoundError(
            "No '*_pca_roi_tc_erpac_results.pkl' files found."
        )

    return files


# ============================================================
# ONLINE STATISTICS
# ============================================================

def update_online_stats(
    accumulator,
    key,
    erpac,
    times,
    amp_freqs,
    subject,
):
    """
    Welford online mean/variance update.

    One observation = one subject ERPAC map.

    accumulator[key] stores:
        n
        mean
        M2
        times
        amp_freqs
        subjects
    """

    erpac = np.asarray(
        erpac,
        dtype=np.float32,
    ).squeeze()

    if erpac.ndim == 1:
        erpac = erpac[
            np.newaxis,
            :
        ]

    times = np.asarray(
        times,
        dtype=np.float32,
    )

    amp_freqs = np.asarray(
        amp_freqs,
        dtype=np.float32,
    )


    expected_shape = (
        len(amp_freqs),
        len(times),
    )

    if erpac.shape != expected_shape:
        raise ValueError(
            f"ERPAC shape mismatch for {key}:\n"
            f"erpac={erpac.shape}, expected={expected_shape}"
        )


    if key not in accumulator:

        accumulator[key] = {
            "n": 0,
            "mean": np.zeros_like(
                erpac,
                dtype=np.float64,
            ),
            "M2": np.zeros_like(
                erpac,
                dtype=np.float64,
            ),
            "times": times,
            "amp_freqs": amp_freqs,
            "subjects": set(),
        }


    stats = accumulator[
        key
    ]


    # Prevent accidental double-counting of one subject.
    if subject in stats[
        "subjects"
    ]:
        raise RuntimeError(
            f"Duplicate subject map detected:\n"
            f"subject={subject}\n"
            f"key={key}"
        )


    if not np.allclose(
        stats["times"],
        times,
    ):
        raise RuntimeError(
            f"Time vector mismatch for {key}"
        )


    if not np.allclose(
        stats["amp_freqs"],
        amp_freqs,
    ):
        raise RuntimeError(
            f"Frequency vector mismatch for {key}"
        )


    stats[
        "subjects"
    ].add(
        subject
    )


    stats[
        "n"
    ] += 1


    n = stats[
        "n"
    ]


    delta = (
        erpac
        - stats[
            "mean"
        ]
    )


    stats[
        "mean"
    ] += (
        delta
        / n
    )


    delta2 = (
        erpac
        - stats[
            "mean"
        ]
    )


    stats[
        "M2"
    ] += (
        delta
        * delta2
    )


def finalize_stats(
    stats,
):
    """
    Return mean, SD, SEM, n.
    """

    n = stats[
        "n"
    ]


    mean = stats[
        "mean"
    ]


    if n > 1:

        variance = (
            stats[
                "M2"
            ]
            / (
                n - 1
            )
        )

        sd = np.sqrt(
            variance
        )

    else:

        sd = np.full_like(
            mean,
            np.nan,
            dtype=np.float64,
        )


    sem = (
        sd
        / np.sqrt(
            n
        )
    )


    return (
        mean,
        sd,
        sem,
        n,
    )


# ============================================================
# READ FILES ONE AT A TIME
# ============================================================

files = find_pkl_files()

print(
    f"Found {len(files)} ERPAC pickle files."
)


# key:
#
# (
#   group,
#   task,
#   stage,
#   block,
#   adaptation_split,
#   condition,
#   coupling,
#   roi
# )
#
accumulator = {}


for idx, file_path in enumerate(
    files,
    start=1,
):

    group = (
        file_path
        .parent
        .parent
        .name
    )

    sub = (
        file_path
        .parent
        .name
    )


    meta = (
        parse_condition_from_filename(
            file_path,
            sub,
        )
    )


    if meta[
        "task"
    ] not in TASKS:
        continue


    print(
        f"[{idx}/{len(files)}] "
        f"{group} | {sub} | "
        f"{meta['condition']}"
    )


    with open(
        file_path,
        "rb",
    ) as f:

        erpac_results = (
            pickle.load(
                f
            )
        )


    for (
        coupling,
        roi_dict,
    ) in erpac_results.items():


        for (
            roi,
            result,
        ) in roi_dict.items():


            if "amp_freqs" not in result:

                raise KeyError(
                    f"'amp_freqs' missing:\n"
                    f"{file_path}\n"
                    f"{coupling} | {roi}"
                )


            key = (
                group,
                meta[
                    "task"
                ],
                meta[
                    "task_stage"
                ],
                meta[
                    "task_block"
                ],
                meta[
                    "adaptation_split"
                ],
                meta[
                    "condition"
                ],
                str(
                    coupling
                ),
                str(
                    roi
                ),
            )


            update_online_stats(
                accumulator=
                    accumulator,

                key=key,

                erpac=result[
                    "erpac"
                ],

                times=result[
                    "times"
                ],

                amp_freqs=result[
                    "amp_freqs"
                ],

                subject=sub,
            )


    # Release this subject-condition immediately.
    del erpac_results

    gc.collect()


print(
    f"\nFinal group maps accumulated: "
    f"{len(accumulator)}"
)


# ============================================================
# GLOBAL FIXED COLOUR SCALE PER COUPLING
# ============================================================
#
# Important:
# These limits are computed from FINAL GROUP-MEAN MAPS,
# not from individual subjects.
#
# ============================================================

coupling_values = {}


for (
    key,
    stats,
) in accumulator.items():

    coupling = key[
        6
    ]

    mean, _, _, _ = (
        finalize_stats(
            stats
        )
    )

    values = mean[
        np.isfinite(
            mean
        )
    ]

    coupling_values.setdefault(
        coupling,
        []
    ).append(
        values.astype(
            np.float32,
            copy=False,
        )
    )


GLOBAL_COLOUR_LIMITS = {}


for (
    coupling,
    value_arrays,
) in coupling_values.items():


    all_values = np.concatenate(
        value_arrays
    )


    if GLOBAL_SCALE_PERCENTILES is None:

        vmin = float(
            np.nanmin(
                all_values
            )
        )

        vmax = float(
            np.nanmax(
                all_values
            )
        )

    else:

        (
            vmin,
            vmax,
        ) = np.nanpercentile(
            all_values,
            GLOBAL_SCALE_PERCENTILES,
        )


        vmin = float(
            vmin
        )

        vmax = float(
            vmax
        )


    GLOBAL_COLOUR_LIMITS[
        coupling
    ] = (
        vmin,
        vmax,
    )


    print(
        f"{coupling}: "
        f"vmin={vmin:.6f}, "
        f"vmax={vmax:.6f}"
    )


    del all_values


# Save scales.
scale_df = pd.DataFrame(
    [
        {
            "coupling": coupling,
            "vmin": limits[0],
            "vmax": limits[1],
            "percentile_low": (
                GLOBAL_SCALE_PERCENTILES[0]
                if GLOBAL_SCALE_PERCENTILES
                is not None
                else np.nan
            ),
            "percentile_high": (
                GLOBAL_SCALE_PERCENTILES[1]
                if GLOBAL_SCALE_PERCENTILES
                is not None
                else np.nan
            ),
        }
        for coupling, limits
        in GLOBAL_COLOUR_LIMITS.items()
    ]
)


scale_df.to_csv(
    OUTPUT_ROOT
    / "erpac_fixed_colour_scales.csv",
    index=False,
)


del coupling_values

gc.collect()


# ============================================================
# PLOTTING HELPERS
# ============================================================

def edges(values):

    values = np.asarray(
        values,
        dtype=float,
    )


    if len(values) == 1:

        return np.array(
            [
                values[0] - 0.5,
                values[0] + 0.5,
            ]
        )


    mids = (
        values[:-1]
        + values[1:]
    ) / 2


    first = (
        values[0]
        - (
            mids[0]
            - values[0]
        )
    )


    last = (
        values[-1]
        + (
            values[-1]
            - mids[-1]
        )
    )


    return np.concatenate(
        [
            [first],
            mids,
            [last],
        ]
    )


def pretty_coupling(
    coupling,
):
    return (
        str(coupling)
        .replace("_", " ")
        .title()
    )


def pretty_condition(
    condition,
):
    return (
        str(condition)
        .replace("_", " | ")
        .replace(
            "plan",
            "Plan",
        )
        .replace(
            "go",
            "Go",
        )
        .replace(
            "baseline",
            "Baseline",
        )
        .replace(
            "adaptation",
            "Adaptation",
        )
        .replace(
            "early",
            "Early",
        )
        .replace(
            "late",
            "Late",
        )
    )


def plot_four_roi(
    group,
    condition,
    coupling,
    roi_stats,
    save_path,
):

    (
        vmin,
        vmax,
    ) = GLOBAL_COLOUR_LIMITS[
        coupling
    ]


    fig, axes = plt.subplots(
        2,
        2,
        figsize=(
            12,
            9,
        ),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )


    axes = axes.ravel()

    mesh = None

    n_subjects = []


    for (
        ax,
        roi,
    ) in zip(
        axes,
        ROI_ORDER,
    ):


        if roi not in roi_stats:

            ax.set_visible(
                False
            )

            continue


        stats = roi_stats[
            roi
        ]


        (
            mean,
            _,
            _,
            n,
        ) = finalize_stats(
            stats
        )


        times = stats[
            "times"
        ]


        freqs = stats[
            "amp_freqs"
        ]


        n_subjects.append(
            n
        )


        mesh = ax.pcolormesh(
            edges(
                times
            ),
            edges(
                freqs
            ),
            mean,
            shading="auto",
            vmin=vmin,
            vmax=vmax,
        )


        if (
            times.min()
            <= 0
            <= times.max()
        ):

            ax.axvline(
                0,
                linestyle="--",
                linewidth=1,
            )


        ax.set_title(
            roi,
            fontsize=14,
        )


        ax.set_xlabel(
            "Time (s)"
        )


        ax.set_ylabel(
            "Gamma (Hz)"
        )


    if mesh is not None:

        cbar = fig.colorbar(
            mesh,
            ax=axes.tolist(),
            shrink=0.82,
            pad=0.04,
        )

        cbar.set_label(
            "ERPAC"
        )


    if len(
        set(
            n_subjects
        )
    ) == 1:

        n_text = (
            f"n = "
            f"{n_subjects[0]}"
        )

    else:

        n_text = (
            "n varies"
        )


    fig.suptitle(
        f"{group} | "
        f"{pretty_condition(condition)} | "
        f"{pretty_coupling(coupling)} | "
        f"{n_text}",
        fontsize=16,
    )


    fig.savefig(
        save_path,
        dpi=DPI,
        bbox_inches="tight",
    )


    plt.close(
        fig
    )


# ============================================================
# ORGANISE FINAL MAP KEYS BY GROUP × CONDITION
# ============================================================

group_condition_keys = {}


for key in accumulator:

    (
        group,
        task,
        stage,
        block,
        adaptation_split,
        condition,
        coupling,
        roi,
    ) = key


    gc_key = (
        group,
        task,
        stage,
        block,
        adaptation_split,
        condition,
    )


    group_condition_keys.setdefault(
        gc_key,
        []
    ).append(
        key
    )


# ============================================================
# SAVE AGGREGATED FILES + FIGURES
# ============================================================

for (
    gc_key,
    keys,
) in group_condition_keys.items():


    (
        group,
        task,
        stage,
        block,
        adaptation_split,
        condition,
    ) = gc_key


    print(
        f"\nSaving "
        f"{group} | {condition}"
    )


    # --------------------------------------------------------
    # Build ONLY one group-condition dataframe at a time.
    # --------------------------------------------------------

    dfs = []


    for key in keys:

        coupling = key[
            6
        ]

        roi = key[
            7
        ]


        stats = accumulator[
            key
        ]


        (
            mean,
            sd,
            sem,
            n,
        ) = finalize_stats(
            stats
        )


        times = stats[
            "times"
        ]


        freqs = stats[
            "amp_freqs"
        ]


        n_freqs = len(
            freqs
        )

        n_times = len(
            times
        )


        df_map = pd.DataFrame(
            {
                "group":
                    group,

                "task":
                    task,

                "task_stage":
                    stage,

                "task_block":
                    block,

                "adaptation_split":
                    adaptation_split,

                "condition":
                    condition,

                "coupling":
                    coupling,

                "roi":
                    roi,

                "amp_freq":
                    np.repeat(
                        freqs,
                        n_times,
                    ),

                "time":
                    np.tile(
                        times,
                        n_freqs,
                    ),

                "mean_erpac":
                    mean.reshape(
                        -1
                    ).astype(
                        np.float32
                    ),

                "sd_erpac":
                    sd.reshape(
                        -1
                    ).astype(
                        np.float32
                    ),

                "sem_erpac":
                    sem.reshape(
                        -1
                    ).astype(
                        np.float32
                    ),

                "n_subjects":
                    np.full(
                        n_freqs
                        * n_times,
                        n,
                        dtype=np.int16,
                    ),
            }
        )


        dfs.append(
            df_map
        )


    condition_df = pd.concat(
        dfs,
        ignore_index=True,
    )


    del dfs


    # --------------------------------------------------------
    # Save data
    # --------------------------------------------------------

    condition_data_dir = (
        AGG_DATA_DIR
        / safe_filename(
            group
        )
        / safe_filename(
            condition
        )
    )


    condition_data_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    condition_df.to_parquet(
        condition_data_dir
        / (
            f"{safe_filename(group)}_"
            f"{safe_filename(condition)}_"
            f"group_erpac.parquet"
        ),
        engine="pyarrow",
        compression="zstd",
        index=False,
    )


    condition_df.to_csv(
        condition_data_dir
        / (
            f"{safe_filename(group)}_"
            f"{safe_filename(condition)}_"
            f"group_erpac.csv"
        ),
        index=False,
    )


    # --------------------------------------------------------
    # Save 4-ROI figures
    # --------------------------------------------------------

    condition_fig_dir = (
        FIG_DIR
        / safe_filename(
            group
        )
        / safe_filename(
            condition
        )
    )


    condition_fig_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    couplings_here = sorted(
        {
            key[
                6
            ]
            for key in keys
        }
    )


    for coupling in couplings_here:

        roi_stats = {
            key[
                7
            ]:
                accumulator[
                    key
                ]

            for key in keys

            if key[
                6
            ] == coupling
        }


        figure_path = (
            condition_fig_dir
            / (
                f"{safe_filename(group)}_"
                f"{safe_filename(condition)}_"
                f"{safe_filename(coupling)}_"
                f"ERPAC_4ROI.png"
            )
        )


        plot_four_roi(
            group=
                group,

            condition=
                condition,

            coupling=
                coupling,

            roi_stats=
                roi_stats,

            save_path=
                figure_path,
        )


    # Release the temporary long dataframe.
    del condition_df

    gc.collect()


# ============================================================
# SAVE SMALL SUMMARY TABLE
# ============================================================

summary_rows = []


for (
    key,
    stats,
) in accumulator.items():

    (
        group,
        task,
        stage,
        block,
        adaptation_split,
        condition,
        coupling,
        roi,
    ) = key


    summary_rows.append(
        {
            "group":
                group,

            "task":
                task,

            "task_stage":
                stage,

            "task_block":
                block,

            "adaptation_split":
                adaptation_split,

            "condition":
                condition,

            "coupling":
                coupling,

            "roi":
                roi,

            "n_subjects":
                stats[
                    "n"
                ],
        }
    )


pd.DataFrame(
    summary_rows
).to_csv(
    OUTPUT_ROOT
    / "group_erpac_summary.csv",
    index=False,
)


print("\n")
print("=" * 70)

print(
    "STREAMING GROUP ERPAC AGGREGATION COMPLETE"
)

print("=" * 70)

print(
    f"\nOutputs saved to:\n"
    f"{OUTPUT_ROOT}"
)


Found 376 ERPAC pickle files.
[1/376] O | s1_pac_sub12 | DeCRAT_go_adaptation_early
[2/376] O | s1_pac_sub12 | DeCRAT_go_adaptation_late
[3/376] O | s1_pac_sub12 | DeCRAT_go_baseline
[4/376] O | s1_pac_sub12 | DeCRAT_plan_adaptation_early
[5/376] O | s1_pac_sub12 | DeCRAT_plan_adaptation_late
[6/376] O | s1_pac_sub12 | DeCRAT_plan_baseline
[7/376] O | s1_pac_sub12 | FTT_go
[8/376] O | s1_pac_sub12 | FTT_plan
[9/376] O | s1_pac_sub14 | DeCRAT_go_adaptation_early
[10/376] O | s1_pac_sub14 | DeCRAT_go_adaptation_late
[11/376] O | s1_pac_sub14 | DeCRAT_go_baseline
[12/376] O | s1_pac_sub14 | DeCRAT_plan_adaptation_early
[13/376] O | s1_pac_sub14 | DeCRAT_plan_adaptation_late
[14/376] O | s1_pac_sub14 | DeCRAT_plan_baseline
[15/376] O | s1_pac_sub14 | FTT_go
[16/376] O | s1_pac_sub14 | FTT_plan
[17/376] O | s1_pac_sub15 | DeCRAT_go_adaptation_early
[18/376] O | s1_pac_sub15 | DeCRAT_go_adaptation_late
[19/376] O | s1_pac_sub15 | DeCRAT_go_baseline
[20/376] O | s1_pac_sub15 | DeCRAT_plan_ada